# Staggered rollouts: practice notebook

**Causal Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

The two-wave example and the simulated green-plate rollout from the module's web page, with `statsmodels`: the one-number regression, fair comparisons by hand, and the event study. Exercises are marked `# YOUR CODE HERE`.

**In Colab:** *File → Save a copy in Drive* first. Run cells with **Shift+Enter**.

## 0. Setup

`greenplate_simulated.csv` is a **simulated** panel shaped like China's green-plate rollout (the real sales data are licensed): 300 cities × 48 months (2015–2018), EV sales, and each city's adoption month. Because the data are made up, we know the true effect: 0.20 log points in the adoption month, growing by 0.055 a month for eight months (half as large in the 5 pilot cities).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/A5/"   # the module's website, e.g. "https://<user>.github.io/<repo>/A5/" (filled in when the site is published)

def load(name):
    for p in [Path(name), Path("data") / name, Path("web/data") / name]:
        if p.exists():
            return pd.read_csv(p)
    return pd.read_csv(SITE + "data/" + name)

d = load("greenplate_simulated.csv")
months = sorted(d.month.unique())
d["t"] = d.month.map({m: k for k, m in enumerate(months)})
d["g"] = d.first_treated.map({m: k for k, m in enumerate(months)})      # adoption month (NaN = not by 2019)
d["y"] = np.log(d.ev_sales + 1)
d.head()

## Part A. Two waves and an effect that grows

Two units observed for 12 periods. The early wave is treated from period 4, the late wave from period 9. The effect is 1 in the first treated period and grows by `growth` each period after.

In [ ]:
def two_waves(growth=1.0):
    t = np.arange(1, 13)
    base = 10 + 0.5 * t
    tauE = np.where(t >= 4, 1 + growth * (t - 4), 0)
    tauL = np.where(t >= 9, 1 + growth * (t - 9), 0)
    toy = pd.DataFrame({"unit": np.repeat(["early", "late"], 12), "t": np.tile(t, 2),
                        "y": np.r_[base + 2 + tauE, base + tauL],
                        "D": np.r_[t >= 4, t >= 9].astype(int), "tau": np.r_[tauE, tauL]})
    return toy

toy = two_waves(1.0)
beta = smf.ols("y ~ D + C(unit) + C(t)", data=toy).fit().params["D"]
print(f"one-number regression {beta:.2f}; true average effect {toy.tau[toy.D == 1].mean():.2f}")

w = toy.pivot(index="t", columns="unit", values="y")
fair = (w.early.loc[4:8].mean() - w.early.loc[1:3].mean()) - (w.late.loc[4:8].mean() - w.late.loc[1:3].mean())
forbidden = (w.late.loc[9:12].mean() - w.late.loc[4:8].mean()) - (w.early.loc[9:12].mean() - w.early.loc[4:8].mean())
print(f"fair (early vs. late, periods 1-8): {fair:.2f}; forbidden (late vs. early, periods 4-12): {forbidden:.2f}")
w.plot(marker="o", title="Two waves"); plt.show()

**Exercise A.** Re-run the regression with `growth=0` (an effect that does not grow) and with `growth=0.5`. When does the one-number regression get the truth?

In [ ]:
# YOUR CODE HERE

## Part B. The simulated green-plate rollout

### B1. Look first: how many cities adopt each month?

In [ ]:
first = d.groupby("city_id").first_treated.first()
print(first.value_counts().sort_index().to_string())
print(d.groupby("city_id").wave.first().value_counts())

### B2. The one-number regression

City and month constants plus a green-plate dummy. Standard errors clustered by city.

In [ ]:
naive = smf.ols("y ~ green_plate + C(city_id) + C(t)", data=d).fit(cov_type="cluster", cov_kwds={"groups": d.city_id})
print(f"one-number regression: {naive.params['green_plate']:.3f} (SE {naive.bse['green_plate']:.3f}); truth: 0.43")

### B3. Fair comparisons by hand (the Callaway–Sant'Anna idea)

For each wave `g` and month `t`: the wave's change since month `g - 1` minus the change of cities **not yet treated** in month `t`. Then average the building blocks, weighting by the number of cities in the wave.

In [ ]:
wide = d.pivot(index="city_id", columns="t", values="y")
G = d.groupby("city_id").g.first()

def block(g, t, rule="not yet"):
    wave = G[G == g].index
    fair = G[G.isna() | (G > t)].index if rule == "not yet" else G[G.isna()].index
    if len(fair) == 0:
        return np.nan
    return (wide.loc[wave, t] - wide.loc[wave, g - 1]).mean() - (wide.loc[fair, t] - wide.loc[fair, g - 1]).mean()

rows = [dict(g=g, e=e, att=block(int(g), int(g) + e), n=(G == g).sum())
        for g in G.dropna().unique() for e in range(-6, 13) if 0 <= g + e < len(months) and e != -1]
blocks = pd.DataFrame(rows).dropna()
es = blocks.groupby("e")[["att", "n"]].apply(lambda s: np.average(s.att, weights=s.n))
post = blocks[blocks.e >= 0]
print(f"fair comparisons, averaged: {np.average(post.att, weights=post.n):.3f}   (one-number regression {naive.params['green_plate']:.3f}; truth 0.43)")
es.plot(marker="o"); plt.axhline(0, color="grey"); plt.axvline(-0.5, ls=":", color="k")
plt.xlabel("months since adoption"); plt.ylabel("effect on log EV sales"); plt.show()

**Exercise B.** (1) Use only never-treated cities as comparisons (`rule="never"`). How many are there, and how do the event study and the average change? (2) Drop the five pilot cities (wave 1) and re-run both the regression and the fair comparisons.

In [ ]:
# YOUR CODE HERE

**Takeaway.** The one-number regression understates the effect because it uses already-treated cities as comparisons while their effects are still growing. Building the estimate only from fair comparisons recovers the truth.